# Baseline Model Training from Snapshot (API, Max Opt)

This notebook builds or reuses a taiko-only constant-BPM snapshot and then trains only the baseline model using the direct Python API.

Flow:
1. Build or reuse a 1000-set snapshot under `C:/taiko-transformer-cache/snapshots`.
2. Create or resume the baseline training context from the snapshot dataset.
3. Train the baseline model and inspect the resulting checkpoint paths.


In [ ]:
from pathlib import Path

import torch

from src.model import (
    ArchitectureSpec,
    TrainingSpec,
    WandbConfig,
    build_training_artifacts,
    create_training_context,
    load_training_context_from_checkpoint,
    train_context,
)
from src.preprocessing import build_snapshot_dataset

repo_root = Path.cwd()
source_unpacked_root = Path("C:/taiko-transformer-cache/unpacked")
snapshot_target_set_count = 1000
snapshot_seed = 42
max_audio_mb = 5.0
snapshot_root = Path(
    f"C:/taiko-transformer-cache/snapshots/taiko_only_static_bpm_{snapshot_target_set_count}_seed{snapshot_seed}"
)
data_root = snapshot_root
training_dir = data_root / "training"
checkpoints_dir = repo_root / "checkpoints" / "baseline_snapshot_maxopt"
last_checkpoint = checkpoints_dir / "last.ckpt"
best_checkpoint = checkpoints_dir / "best.ckpt"

index_cache_dir = training_dir / "index_cache"
inference_snapshots_dir = checkpoints_dir / "inference_snapshots"

epochs = 10
batch_size = 16
num_workers = 0
precision = "auto"
pin_memory = True
persistent_workers = False
prefetch_factor = 2
architecture_name = "taiko_transformer"
keep_only_max_notes_per_song = True
build_snapshot = False
overwrite_snapshot = False
save_inference_every_n_steps = 1000
run_name = "baseline_snapshot_maxopt"
use_resume_if_available = True
use_wandb = False
wandb_log_every_batches = 100
wandb_notebook_name = "train_baseline_snapshot_api_maxopt.ipynb"
wandb_api_key = ""
wandb_offline = False

if torch.cuda.is_available():
    best_device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    best_device = "mps"
else:
    best_device = "cpu"

checkpoints_dir.mkdir(parents=True, exist_ok=True)

print(f"repo_root                     : {repo_root}")
print(f"source_unpacked_root          : {source_unpacked_root}")
print(f"snapshot_root                 : {snapshot_root}")
print(f"data_root                     : {data_root}")
print(f"training_dir                  : {training_dir}")
print(f"checkpoints_dir               : {checkpoints_dir}")
print(f"last_checkpoint               : {last_checkpoint}")
print(f"index_cache_dir               : {index_cache_dir}")
print(f"inference_snapshots_dir       : {inference_snapshots_dir}")
print(f"best_checkpoint               : {best_checkpoint}")
print(f"snapshot_target_set_count     : {snapshot_target_set_count}")
print(f"keep_only_max_notes_per_song  : {keep_only_max_notes_per_song}")
print(f"best_device                   : {best_device}")


## Step 1: Build or reuse the snapshot dataset


In [ ]:
snapshot_summary = None
required_snapshot_files = [
    snapshot_root / "chart_index" / "chart_build_summary.csv",
    snapshot_root / "beat_aligned_dataset" / "sequence_metadata.csv",
]
snapshot_ready = all(path.exists() for path in required_snapshot_files)
should_build_snapshot = build_snapshot or not snapshot_ready

if should_build_snapshot:
    if not snapshot_ready and not build_snapshot:
        print("Snapshot dataset artifacts were not found; building the snapshot automatically.")
    elif build_snapshot:
        print("Building snapshot dataset because build_snapshot=True.")
    snapshot_summary = build_snapshot_dataset(
        source_unpacked_root=source_unpacked_root,
        snapshot_root=snapshot_root,
        target_set_count=snapshot_target_set_count,
        seed=snapshot_seed,
        max_audio_mb=max_audio_mb,
        overwrite=overwrite_snapshot,
        keep_only_max_notes_per_song=keep_only_max_notes_per_song,
    )
    print(snapshot_summary)
else:
    print("Skipping snapshot build and reusing existing snapshot dataset.")

baseline_artifacts = build_training_artifacts(data_root, checkpoints_dir=checkpoints_dir)
print(baseline_artifacts)


## Step 2: Create or resume the baseline training context


In [ ]:
baseline_architecture_spec = ArchitectureSpec(name=architecture_name)

baseline_training_spec = TrainingSpec(
    epochs=epochs,
    batch_size=batch_size,
    num_workers=num_workers,
    device=best_device,
    precision=precision,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    prefetch_factor=prefetch_factor,
)

wandb_config = None
if use_wandb:
    wandb_config = WandbConfig(
        enabled=True,
        run_name=run_name,
        log_every_n_batches=wandb_log_every_batches,
        notebook_name=wandb_notebook_name,
        offline=wandb_offline,
        api_key=wandb_api_key,
        mode_name_for_run=architecture_name,
    )

if use_resume_if_available and last_checkpoint.exists():
    baseline_context = load_training_context_from_checkpoint(
        last_checkpoint,
        data_root=data_root,
        device=best_device,
        batch_size=batch_size,
        num_workers=num_workers,
        checkpoints_dir=checkpoints_dir,
        index_cache_dir=index_cache_dir,
        precision=precision,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
    )
    print(f"Resuming from checkpoint: {last_checkpoint}")
else:
    baseline_context = create_training_context(
        data_root=data_root,
        architecture_spec=baseline_architecture_spec,
        training_spec=baseline_training_spec,
        checkpoints_dir=checkpoints_dir,
        index_cache_dir=index_cache_dir,
        precision=precision,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
    )
    print("Starting a fresh baseline-model snapshot training run.")

print("baseline architecture:", baseline_context.architecture_spec)
print("baseline ignore_index:", baseline_context.dataset.label_ignore_index)
print("start_epoch:", baseline_context.start_epoch)
print("target_epochs:", epochs)


## Step 3: Train the baseline model


In [ ]:
baseline_context = train_context(
    baseline_context,
    epochs=epochs,
    log_every_n_batches=wandb_log_every_batches,
    wandb_config=wandb_config,
    save_inference_every_n_steps=save_inference_every_n_steps,
    inference_snapshots_dir=inference_snapshots_dir,
)

print("Training finished.")
print(f"last checkpoint: {last_checkpoint.resolve()}")
print(f"best checkpoint: {best_checkpoint.resolve()}")
print(f"inference snapshots dir: {inference_snapshots_dir.resolve()}")


## Optional inspection


In [ ]:
print("history keys:", baseline_context.history.keys())
print("snapshot root:", snapshot_root)
print("training dir:", training_dir)
print("selection manifest:", snapshot_root / "selection_manifest.csv")
print("rejection report:", snapshot_root / "rejection_report.csv")
print("vocab json:", training_dir / "vocab.json")
print("splits json:", training_dir / "splits.json")
